In [ ]:
!pip install dash dash-bootstrap-components

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# Asegurar renderizado correcto en Colab
pio.renderers.default = "colab"

# ------------------------------------------------------------------------------
# 1. GENERACIÓN DE DATOS DE COCA-COLA
# ------------------------------------------------------------------------------
np.random.seed(101)

dates = pd.date_range(start="2022-01-02", end="2022-12-25", freq="D")
retailers = ["Walmart", "CVS", "Costco", "Target"]
brands = ["Coca-Cola", "Dasani Water", "Diet Coke", "Sprite", "Powerade", "Fanta"]

states = {
    'NY': (40.7128, -74.0060, 'New York'), 'CA': (36.7783, -119.4179, 'California'),
    'TX': (31.9686, -99.9018, 'Texas'), 'FL': (27.6648, -81.5158, 'Florida'),
    'IL': (40.6331, -89.3985, 'Illinois'), 'PA': (41.2033, -77.1945, 'Pennsylvania'),
    'OH': (40.4173, -82.9071, 'Ohio'), 'GA': (32.1656, -82.9001, 'Georgia')
}

data = []
for d in dates:
    r = np.random.choice(retailers, p=[0.45, 0.25, 0.20, 0.10])
    b = np.random.choice(brands, p=[0.25, 0.20, 0.18, 0.15, 0.12, 0.10])
    st_code = np.random.choice(list(states.keys()))
    units = np.random.randint(1500, 6000)
    price = round(np.random.uniform(0.35, 0.55), 2)
    sales = units * price
    profit = sales * round(np.random.uniform(0.35, 0.40), 2)

    data.append({
        'Date': d, 'Quarter': f"Qtr {d.quarter}", 'Retailer': r, 'Beverage Brand': b,
        'State': states[st_code][2], 'Lat': states[st_code][0], 'Lon': states[st_code][1],
        'Units Sold': units, 'Price per Unit': price, 'Total Sales': sales, 'Operating Profit': profit
    })

df = pd.DataFrame(data)

# ------------------------------------------------------------------------------
# 2. CONSTRUCCIÓN DE LAS 4 VISUALIZACIONES
# ------------------------------------------------------------------------------

# --- 1. Gráfico de Barras por Retailer ---
ret_sum = df.groupby('Retailer')[['Total Sales', 'Operating Profit']].sum().reset_index()
fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(x=ret_sum['Retailer'], y=ret_sum['Total Sales'] - ret_sum['Operating Profit'], name='Sales', marker_color='#3399ff'))
fig_bar.add_trace(go.Bar(x=ret_sum['Retailer'], y=ret_sum['Operating Profit'], name='Operating Profit', marker_color='#9370db'))
fig_bar.update_layout(
    barmode='stack', title='Total Sales and Operating Profit by Retailer',
    height=320, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", y=1.1)
)

# --- 2. Tabla Interactiva de Marcas ---
brand_summary = df.groupby('Beverage Brand').agg(
    Total_Sales=('Total Sales', 'sum'),
    Units_Sold=('Units Sold', 'sum'),
    Operating_Profit=('Operating Profit', 'sum')
).reset_index()

fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(brand_summary.columns), fill_color='#0b1340', align='left', font=dict(color='white', size=12)),
    cells=dict(values=[
        brand_summary['Beverage Brand'],
        brand_summary['Total_Sales'].map('${:,.0f}'.format),
        brand_summary['Units_Sold'].map('{:,.0f}'.format),
        brand_summary['Operating_Profit'].map('${:,.0f}'.format)
    ], fill_color='#f4f6f8', align='left', font=dict(size=11))
)])
fig_table.update_layout(title='Beverage Brand Performance', height=320, margin=dict(l=10, r=10, t=40, b=10))

# --- 3. Línea de Tendencia por Trimestre ---
qtr_sum = df.groupby('Quarter')[['Total Sales', 'Operating Profit']].sum().reset_index()
fig_line = go.Figure()
fig_line.add_trace(go.Scatter(x=qtr_sum['Quarter'], y=qtr_sum['Total Sales'], mode='lines+markers', name='Sales', line=dict(color='#0080ff', width=3)))
fig_line.add_trace(go.Scatter(x=qtr_sum['Quarter'], y=qtr_sum['Operating Profit'], mode='lines+markers', name='Profit', line=dict(color='#110066', width=3)))
fig_line.update_layout(
    title='Operating Profit Analysis by Quarter',
    height=320, margin=dict(l=20, r=20, t=40, b=20), legend=dict(orientation="h", y=1.1)
)

# --- 4. Mapa Interactivo de EE. UU. ---
st_sum = df.groupby(['State', 'Lat', 'Lon'])['Total Sales'].sum().reset_index()
fig_map = go.Figure(data=go.Scattergeo(
    lon=st_sum['Lon'],
    lat=st_sum['Lat'],
    text=st_sum['State'] + ': $' + st_sum['Total Sales'].map('{:,.0f}'.format),
    mode='markers',
    marker=dict(
        size=st_sum['Total Sales'] / st_sum['Total Sales'].max() * 25 + 8,
        color='#3399ff', opacity=0.8, line=dict(width=1, color='blue')
    )
))
fig_map.update_layout(
    title='Sales by US State',
    geo=dict(scope='usa', projection_type='albers usa', showland=True, landcolor="rgb(240, 240, 240)"),
    height=320, margin=dict(l=0, r=0, t=40, b=0)
)

# ------------------------------------------------------------------------------
# 3. ACOMODO EN PANTALLA ÚNICA (CUADRÍCULA 2x2)
# ------------------------------------------------------------------------------
out1, out2, out3, out4 = widgets.Output(), widgets.Output(), widgets.Output(), widgets.Output()

with out1:
    fig_bar.show()
with out2:
    fig_table.show()
with out3:
    fig_line.show()
with out4:
    fig_map.show()

grid = widgets.GridBox(
    children=[out1, out2, out3, out4],
    layout=widgets.Layout(
        display='flex',
        flex_flow='row wrap',
        justify_content='space-between'
    )
)

header = widgets.HTML("<h2 style='color:#0b1340; text-align:center;'>Coca-Cola Sales Dashboard</h2><hr>")
display(widgets.VBox([header, grid]))

Support for third party widgets will remain active for the duration of the session. To disable support: